# E1.8 · Third-party and model supply chain risk

**Function E — Governance, Risk, Compliance & the CISO Office → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

---

**Risk.** Vendor AI features enabled by default; sub-processor chains you never mapped.

**Control.** Questions that actually discriminate between vendors.

**This lab.** Run a real AIBOM against a model artefact.

| | |
|---|---|
| Open-source tooling | OWASP AIBOM, Sigstore |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E1.8"))

Third-party and model supply-chain risk. Two of the artefacts have no mature provenance story yet, and saying so is part of the assessment.

In [ ]:
from cybercommons import research
P = research.Package

for p in [P("langchain-community", "0.2.1", signed=False, downloads=400_000, age_days=200),
          P("mcp-jira-connector",  "0.0.3", signed=False, downloads=90,      age_days=6),
          P("cryptography",        "42.0.5", signed=True,  downloads=900_000, age_days=300)]:
    r = research.provenance(p)
    print(f"{r['package']:30s}{r['verdict']}")
    for f in r["flags"]:
        print(f"      · {f}")

Now the two artefacts your existing process does not cover.

In [ ]:
GAPS = {
 "model weights":   ("Sigstore/in-toto attestation is possible and rare",
                     "no download-count equivalent; 'popular checkpoint' is not provenance"),
 "prompt/tool packs": ("no signing convention at all",
                     "runs inside the agent with the agent's authority"),
 "the model API itself": ("version can change server-side without notice",
                     "your pinned dependency list does not include it"),
}
for artefact, (state, why) in GAPS.items():
    print(f"{artefact}\n   state: {state}\n   why it matters: {why}\n")

### Expect

The mature package is allowed, the new unsigned MCP connector is blocked or flagged for review, and the three gap areas print with an honest statement of what does not yet exist.

### Your turn

Add one question to your third-party assessment: 'can the model version change without notifying us?' The answer is usually yes, and it changes the risk rating.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E1.8.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*